In [1]:
from pathlib import Path

import pandas as pd

from mlxtend.frequent_patterns import fpgrowth
from mlxtend.frequent_patterns import association_rules

In [2]:
DATA_DIR = Path("../data")

PROCESSED_DIR = DATA_DIR / "processed"

TRANSACTIONS_PATH = PROCESSED_DIR / "transactions.parquet"
ARTICLES_PATH = PROCESSED_DIR / "articles.parquet"

In [3]:
transactions = pd.read_parquet(
    TRANSACTIONS_PATH, 
    columns=["customer_id", 
             "article_id"]
)

articles = pd.read_parquet(
    ARTICLES_PATH
)

demo_users = pd.read_parquet(
    PROCESSED_DIR / "demo_users.parquet"
)

transactions = transactions[
    transactions["customer_id"].isin(
        demo_users["customer_id"]
    )
]

TOP_PRODUCTS = 3000

popular_products = (
    transactions["article_id"]
    .value_counts()
    .head(TOP_PRODUCTS)
    .index
)

transactions = transactions[
    transactions["article_id"].isin(popular_products)
]

print(transactions.shape)

(206538, 2)


In [4]:
product_counts = (
    transactions
    .groupby("article_id")
    .size()
    .reset_index(name="purchases")
)

product_counts.head()

,article_id,purchases
0,108775015,208
1,108775044,117
2,111565001,61
3,111586001,202
4,111593001,230


In [5]:
MIN_PURCHASES = 10

popular_products = product_counts.loc[
    product_counts["purchases"] >= MIN_PURCHASES,
    "article_id"
]

transactions = transactions[
    transactions["article_id"]
    .isin(popular_products)
]

In [6]:
basket = (
    transactions.assign(value=1)
    .pivot_table(
        index="customer_id",
        columns="article_id",
        values="value",
        fill_value=0
    )
).astype(bool)

In [7]:
frequent_itemsets = fpgrowth(
    basket,
    min_support=0.002,
    use_colnames=True
)

In [24]:
rules = association_rules(
    frequent_itemsets,
    metric="lift",
    min_threshold=1.0
)

In [25]:
rules = rules[
    (rules["antecedents"].apply(len) == 1)
    &
    (rules["consequents"].apply(len) == 1)
].copy()

In [26]:
rules["article_id"] = (
    rules["antecedents"]
    .apply(lambda x: list(x)[0])
)

rules["recommended_article_id"] = (
    rules["consequents"]
    .apply(lambda x: list(x)[0])
)

In [27]:
rules = rules[
    [
        "article_id",
        "recommended_article_id",
        "support",
        "confidence",
        "lift"
    ]
]

In [28]:
product_names = articles[
    [
        "article_id",
        "prod_name"
    ]
]

In [29]:
rules = rules.merge(
    product_names,
    on="article_id",
    how="left"
)

rules = rules.rename(
    columns={
        "prod_name": "product_name"
    }
)

In [30]:
recommended_names = (
    product_names.rename(
        columns={
            "article_id": "recommended_article_id",
            "prod_name": "recommended_product_name"
        }
    )
)

In [31]:
rules = rules.merge(
    recommended_names,
    on="recommended_article_id",
    how="left"
)

In [32]:
product_groups = articles[
    [
        "article_id",
        "product_group_name",
        "product_type_name"
    ]
]

In [33]:
rules = rules.merge(
    product_groups.rename(
        columns={
            "product_group_name": "source_product_group",
            "product_type_name": "source_product_type"
        }
    ),
    on="article_id",
    how="left"
)

In [34]:
rules = rules.merge(
    product_groups.rename(
        columns={
            "article_id": "recommended_article_id",
            "product_group_name": "target_product_group",
            "product_type_name": "target_product_type"
        }
    ),
    on="recommended_article_id",
    how="left"
)

In [35]:
rules = rules[
    rules["source_product_type"]
    != rules["target_product_type"]
]

In [36]:
rules = rules.sort_values(
    ["article_id", "lift"],
    ascending=[True, False]
)

In [37]:
rules_final = rules.groupby("article_id").head(10)

In [38]:
rules_final

,article_id,recommended_article_id,support,confidence,lift,product_name,recommended_product_name,source_product_group,source_product_type,target_product_group,target_product_type
4989,108775015,768912001,0.0020,0.074074,3.165559,Strap top,Long leggings update,Garment Upper body,Vest top,Garment Lower body,Leggings/Tights
4819,108775015,179123001,0.0020,0.074074,2.743484,Strap top,Long Leggings,Garment Upper body,Vest top,Garment Lower body,Leggings/Tights
4831,108775015,536139006,0.0022,0.081481,2.425044,Strap top,Alex Jogger (J),Garment Upper body,Vest top,Nightwear,Pyjama bottom
4841,108775015,554450001,0.0024,0.088889,2.415459,Strap top,Julia RW Skinny Denim TRS,Garment Upper body,Vest top,Garment Lower body,Trousers
4807,108775015,591334003,0.0020,0.074074,2.344116,Strap top,Flock (1),Garment Upper body,Vest top,Garment Upper body,Sweater
...,...,...,...,...,...,...,...,...,...,...,...
8424,880839001,706016001,0.0020,0.120482,1.231922,Eleonor button dress,Jade HW Skinny Denim TRS,Garment Full body,Dress,Garment Lower body,Trousers
9341,883033002,827968001,0.0020,0.123457,4.938272,CS Paula dress,Antonia heavy t-shirt,Garment Full body,Dress,Garment Upper body,T-shirt
9343,883033002,706016001,0.0022,0.135802,1.388573,CS Paula dress,Jade HW Skinny Denim TRS,Garment Full body,Dress,Garment Lower body,Trousers
8806,915292001,866731001,0.0028,0.466667,33.816425,Lana seamless cropped ls,LANA seamless HW tigths,Garment Upper body,Top,Garment Lower body,Leggings/Tights


In [39]:
rules_final.to_parquet(
    PROCESSED_DIR /
    "bought_together.parquet",
    index=False
)